In [1]:
#KERNAL
# TERMINAL:
# conda activate azureml_py310_sdkv2
# pip install --upgrade pip
# pip install dotenv
# pip install azure-search-documents==11.6.0b12 azure-identity openai tiktoken aiohttp ipykernel requests --quiet

In [6]:
import pandas as pd
import numpy as np

In [7]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Load environment variables from .env
load_dotenv("globalparameters.env")

# Azure Storage
STORAGE_ACCOUNT_NAME = os.getenv("STORAGE_ACCOUNT_NAME")
STORAGE_ACCOUNT_KEY = os.getenv("STORAGE_ACCOUNT_KEY")
BLOB_CONTAINER_NAME = os.getenv("BLOB_CONTAINER_NAME")
CONNECTION_STRING = f"DefaultEndpointsProtocol=https;AccountName={STORAGE_ACCOUNT_NAME};AccountKey={STORAGE_ACCOUNT_KEY};EndpointSuffix=core.windows.net"  

# Azure Search
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME = os.getenv("INDEX_NAME")
SKILLSET_NAME = os.getenv("SKILLSET_NAME")

# Azure OpenAI
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_VERSION = os.getenv("AZURE_OPENAI_VERSION")
AZURE_OPENAI_GPT_DEPLOYMENT = os.getenv("AZURE_OPENAI_GPT_DEPLOYMENT")
AZURE_OPENAI_GPT_MODEL = os.getenv("AZURE_OPENAI_GPT_MODEL")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
AZURE_OPENAI_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
AZURE_OPENAI_EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", 1536))

# Azure Cognitive Services 
AZURE_AI_MULTISERVICES_ENDPOINT = os.getenv("AZURE_AI_MULTISERVICES_ENDPOINT")
AZURE_AI_MULTISERVICES_KEY = os.getenv("AZURE_AI_MULTISERVICES_KEY")

# Agent Name
AGENT_NAME =  os.getenv("AGENT_NAME")
OPENAI_ANSWER_MODEL =  os.getenv("OPENAI_ANSWER_MODEL")
OPENAI_ANSWER_MODEL_API_VERSION =  os.getenv("OPENAI_ANSWER_MODEL_API_VERSION")

ModuleNotFoundError: No module named 'dotenv'

## Helper functions - Create Search Index and upload documents

In [4]:
# Azure Search Imports
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SimpleField,
    FieldMapping,
    FieldMappingFunction,
    SearchFieldDataType,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SearchIndexerDataSourceConnection,
    SearchIndexerDataContainer,
    SearchIndexer,
    IndexingParameters,
    IndexingParametersConfiguration,
    BlobIndexerParsingMode,
    BlobIndexerImageAction,
    SplitSkill,
    InputFieldMappingEntry,
    OutputFieldMappingEntry,
    AzureOpenAIEmbeddingSkill,
    EntityRecognitionSkill,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    IndexProjectionMode,
    SearchIndexerSkillset,
    AIServicesAccountKey,
    AIServicesAccountIdentity
)

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential

# Azure Storage Blob
from azure.storage.blob import BlobServiceClient

# OpenAI
import openai
from openai import OpenAIError

# Python built-ins and third party
import os
import re
import tempfile
from io import StringIO
import uuid

########################################################################################################################################################################################################
# Get Azure Search Credentail
########################################################################################################################################################################################################
def get_azure_credential(api_key: str, use_aad: bool = False):
    """
    Returns Azure credential for Search API authentication.
    Args:
        api_key (str): Azure Search API key.
        use_aad (bool): Whether to use Azure Active Directory authentication. Defaults to False.

    Returns:
        AzureKeyCredential or DefaultAzureCredential
    """
    if use_aad:
        print("Using AAD for authentication.")
        return DefaultAzureCredential()
    else:
        print("Using API key authentication.")
        if not api_key:
            raise ValueError("API key must be provided if not using AAD authentication.")
        
        return AzureKeyCredential(api_key) if api_key else DefaultAzureCredential()

########################################################################################################################################################################################################
# Create Blob Data Source Connector on AI Search Indexer
########################################################################################################################################################################################################
def create_blob_datasource_connection(index_name: str, credential, connection_string: str, container_name: str, search_endpoint: str) -> str:
    """
    Creates or updates a blob data source connection for an Azure Search indexer.

    Args:
        index_name (str): The name of the search index.
        credential: Azure credential (AzureKeyCredential or DefaultAzureCredential).
        connection_string (str): Azure Blob Storage connection string.
        container_name (str): Blob container name.
        search_endpoint (str): Azure Search service endpoint.

    Returns:
        str: The name of the data source connection.
    """
    indexer_client = SearchIndexerClient(endpoint=search_endpoint, credential=credential)
    container = SearchIndexerDataContainer(name=container_name)

    data_source = SearchIndexerDataSourceConnection(
        name=f"{index_name}-blob",
        type="azureblob",
        connection_string=connection_string,
        container=container,
    )

    print(f"Creating data source connection: {data_source}")

    try:
        indexer_client.create_or_update_data_source_connection(data_source)
        print(f"Data source '{data_source.name}' created or updated successfully.")
    except Exception as err:
        raise RuntimeError(f"Failed to create/update data source connection: {err}")

    return data_source.name

########################################################################################################################################################################################################
# Create the index (if not already created)
########################################################################################################################################################################################################
def create_search_index(azure_search_endpoint, 
                        azure_credential, 
                        azure_openai_endpoint, 
                        azure_openai_embedding_deployment, 
                        azure_openai_embedding_model,
                        azure_openai_model_dimensions,
                        azure_openai_key):
    index_client = SearchIndexClient(endpoint=azure_search_endpoint, credential=azure_credential)

    # Delete index if it already exists
    if INDEX_NAME in [idx.name for idx in index_client.list_indexes()]:
        index_client.delete_index(INDEX_NAME)
    fields = [
            #Search Fields
            SearchField(name="AzureSearch_DocumentKey",  key=True, type=SearchFieldDataType.String),
            SearchField(name="Loan_Number", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=True), 
            SearchField(name="Borrower", type=SearchFieldDataType.String, key=False,sortable=True, filterable=True, facetable=True), 
            SearchField(name="LoanNumberVector", 
                        type=SearchFieldDataType.Collection(SearchFieldDataType.Single), 
                        vector_search_dimensions=azure_openai_model_dimensions, 
                        vector_search_profile_name="hnsw_text_3_small"),
            SearchField(name="BorrowerVector", 
                        type=SearchFieldDataType.Collection(SearchFieldDataType.Single), 
                        vector_search_dimensions=azure_openai_model_dimensions, 
                        vector_search_profile_name="hnsw_text_3_small"),            
            SearchField(name="Borrower_2", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=False),
            SearchField(name="Borrower_3", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=False),
            SearchField(name="Borrower_4", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=False),
            SearchField(name="Borrowers", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=False),

            #Simple Fields
            SimpleField(name="City", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=True),
            SimpleField(name="State", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=True),
            SimpleField(name="Date_af_date", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=True),
            SimpleField(name="LoanAmountText", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=True),
            SimpleField(name="InterestRate", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=True),   
            SimpleField(name="Borrower1Initials", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=False),
            SimpleField(name="Borrower2Initials", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=False),
            SimpleField(name="Borrower3Initials", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=False),
            SimpleField(name="Borrower4Initials", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=False),
            SimpleField(name="RelationshiptoCoborrower", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=True),
            SimpleField(name="RelationshiptoBorrower", type=SearchFieldDataType.String, key=False, sortable=True, filterable=True, facetable=True),    
    ] 

    vector_search=VectorSearch(
            profiles=[
                VectorSearchProfile(
                    name="hnsw_text_3_small",
                    algorithm_configuration_name="myHnsw",
                    vectorizer_name="azure_openai_text_3_small"
                )
            ],
            algorithms=[HnswAlgorithmConfiguration(name="myHnsw")],
            vectorizers=[
                AzureOpenAIVectorizer(
                    vectorizer_name="azure_openai_text_3_small",
                    kind="azureOpenAI",  
                    parameters=AzureOpenAIVectorizerParameters(
                        resource_url=azure_openai_endpoint,
                        deployment_name=azure_openai_embedding_deployment,
                        model_name=azure_openai_embedding_model,
                        api_key=azure_openai_key
                    )
                )
            ]
        ) 

    semantic_config = SemanticConfiguration(  
        name="my-semantic-config",  
        prioritized_fields=SemanticPrioritizedFields(  
            content_fields=[SemanticField(field_name="Borrower")],
            title_field=SemanticField(field_name="Loan_Number")
        ),  
    )
  
    # Create the semantic search with the configuration   
    semantic_search = SemanticSearch(configurations=[semantic_config], default_configuration_name="my-semantic-config") #Enable default configuration name for agentic retrival

    # Create the search index
    index = SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vector_search, semantic_search=semantic_search)  
    result = index_client.create_or_update_index(index)  
    print(f"{result.name} created")  

########################################################################################################################################################################################################
# Create a skillset
########################################################################################################################################################################################################
def create_permit_search_markdown_skillset(
    skillset_name: str,
    index_name: str,
    azure_openai_endpoint: str,
    azure_openai_embedding_deployment: str,
    azure_openai_embedding_model: str,
    azure_openai_model_dimensions: str,
    azure_search_endpoint: str,
    azure_search_credential,
):    
    # Azure OpenAI Embedding Skill
    loannumber_embedding_skill = AzureOpenAIEmbeddingSkill(
        description="Skill to generate embeddings via Azure OpenAI",
        context="/document",
        resource_url=azure_openai_endpoint,
        deployment_name=azure_openai_embedding_deployment,
        model_name=azure_openai_embedding_model,
        dimensions=azure_openai_model_dimensions,
        inputs=[InputFieldMappingEntry(name="text", source="/document/Loan_Number")],
        outputs=[OutputFieldMappingEntry(name="embedding", target_name="LoanNumberVector")]
    )

    borrower_embedding_skill = AzureOpenAIEmbeddingSkill(
        description="Skill to generate embeddings via Azure OpenAI",
        context="/document",
        resource_url=azure_openai_endpoint,
        deployment_name=azure_openai_embedding_deployment,
        model_name=azure_openai_embedding_model,
        dimensions=azure_openai_model_dimensions,
        inputs=[InputFieldMappingEntry(name="text", source="/document/Borrower")],
        outputs=[OutputFieldMappingEntry(name="embedding", target_name="BorrowerVector")]
    )

    skills = [loannumber_embedding_skill, borrower_embedding_skill]

    # Compose the Skillset
    skillset = SearchIndexerSkillset(
        name=skillset_name,
        description="Skillset to generate embeddings",
        skills = skills
    )

    # Create or update the skillset in Azure Search
    client = SearchIndexerClient(endpoint=azure_search_endpoint, credential=azure_search_credential)
    client.create_or_update_skillset(skillset)
    print(f"Skillset '{skillset.name}' created or updated successfully.")
   
########################################################################################################################################################################################################
# Create Indexer
########################################################################################################################################################################################################
def create_indexer(index_name, data_source_name, search_service_endpoint, azure_search_credential):
   # Create an indexer  
    indexer_name = f"{index_name}-indexer"  

    indexer_parameters = IndexingParameters(
        configuration=IndexingParametersConfiguration(
            parsing_mode=BlobIndexerParsingMode.DELIMITED_TEXT,
            query_timeout=None,
            first_line_contains_headers=True)) 

    indexer = SearchIndexer(  
        name=indexer_name,  
        description="Indexer to index documents and generate embeddings",  
        target_index_name=index_name,  
        data_source_name=data_source_name,
        parameters=indexer_parameters,
        field_mappings=[FieldMapping(source_field_name="AzureSearch_DocumentKey", target_field_name="AzureSearch_DocumentKey", mapping_function=FieldMappingFunction(name="base64Encode"))],
        output_field_mappings=[
            FieldMapping(source_field_name="/document/LoanNumberVector", target_field_name="LoanNumberVector"),
            FieldMapping(source_field_name="/document/BorrowerVector", target_field_name="BorrowerVector")
        ]
    )  

    indexer_client = SearchIndexerClient(search_service_endpoint, azure_search_credential)  
    indexer_result = indexer_client.create_or_update_indexer(indexer)  
  
    # Run the indexer  
    indexer_client.run_indexer(indexer_name)  
    print(f' {indexer_name} is created and running. If queries return no results, please wait a bit and try again.')  

### Create Search Indexer

In [5]:
# Step 1: Get azure search credential
azure_search_credential = get_azure_credential(api_key = AZURE_SEARCH_API_KEY)

Using API key authentication.


In [6]:
#Step 2: Create blob data source connection
data_source_connection_name = create_blob_datasource_connection(index_name = INDEX_NAME, 
                                                                credential= azure_search_credential,
                                                                connection_string= CONNECTION_STRING,
                                                                container_name= BLOB_CONTAINER_NAME,
                                                                search_endpoint= AZURE_SEARCH_ENDPOINT)

Creating data source connection: {'additional_properties': {}, 'name': 'condoai-applicant-datastore-searchindex-v9-blob', 'description': None, 'type': 'azureblob', 'connection_string': 'DefaultEndpointsProtocol=https;AccountName=mdcprojectstot7kds;AccountKey=ENINz+cvHHMP2dMjIzCvybSm4PF3vdys8Clq/xp0g0u1f5/EHwvMjaRLeuBZO6it3smcCTDOIx1E+AStAkqh1A==;EndpointSuffix=core.windows.net', 'container': <azure.search.documents.indexes._generated.models._models_py3.SearchIndexerDataContainer object at 0x7969de64f9a0>, 'identity': None, 'indexer_permission_options': None, 'data_change_detection_policy': None, 'data_deletion_detection_policy': None, 'e_tag': None, 'encryption_key': None}
Data source 'condoai-applicant-datastore-searchindex-v9-blob' created or updated successfully.


In [7]:
# Step 3: Create Search Index
create_search_index(azure_search_endpoint=AZURE_SEARCH_ENDPOINT, 
                    azure_credential=azure_search_credential, 
                    azure_openai_endpoint=AZURE_OPENAI_ENDPOINT, 
                    azure_openai_embedding_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    azure_openai_embedding_model=AZURE_OPENAI_EMBEDDING_MODEL, 
                    azure_openai_model_dimensions=AZURE_OPENAI_EMBEDDING_DIMENSIONS,
                    azure_openai_key=AZURE_OPENAI_KEY )

condoai-applicant-datastore-searchindex-v9 created


In [8]:
#STEP 4: Created skillset for markdown files
create_permit_search_markdown_skillset(skillset_name = SKILLSET_NAME, 
                                       index_name = INDEX_NAME, 
                                       azure_openai_endpoint = AZURE_OPENAI_ENDPOINT, 
                                       azure_openai_embedding_deployment = AZURE_OPENAI_EMBEDDING_DEPLOYMENT, 
                                       azure_openai_embedding_model = AZURE_OPENAI_EMBEDDING_MODEL, 
                                       azure_search_endpoint = AZURE_SEARCH_ENDPOINT,
                                       azure_openai_model_dimensions = AZURE_OPENAI_EMBEDDING_DIMENSIONS, 
                                       azure_search_credential= azure_search_credential, )

Skillset 'condoai-applicant-datastore-skillset-v9' created or updated successfully.


In [9]:
# Step 5: Create an Indexer
create_indexer(index_name = INDEX_NAME, 
               data_source_name = data_source_connection_name, 
               search_service_endpoint= AZURE_SEARCH_ENDPOINT, 
               azure_search_credential = azure_search_credential)

 condoai-applicant-datastore-searchindex-v9-indexer is created and running. If queries return no results, please wait a bit and try again.


Create a Knowledge Agent for retriving Individual applicants details

In [10]:
from azure.search.documents.indexes.models import KnowledgeAgent, KnowledgeAgentAzureOpenAIModel, KnowledgeAgentTargetIndex, KnowledgeAgentRequestLimits, AzureOpenAIVectorizerParameters

agent = KnowledgeAgent(
    name=AGENT_NAME,
    models=[
        KnowledgeAgentAzureOpenAIModel(
            azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                resource_url=AZURE_OPENAI_ENDPOINT,
                deployment_name=OPENAI_ANSWER_MODEL,
                model_name=OPENAI_ANSWER_MODEL
            )
        )
    ],
    target_indexes=[
        KnowledgeAgentTargetIndex(
            index_name=INDEX_NAME,
            default_semantic_configuration_name="my-semantic-config",
            default_reranker_threshold=2.5
        )
    ],
)

# Instantiate SearchIndexClient
index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    credential=azure_search_credential
)
index_client.create_or_update_agent(agent)
print(f"Knowledge agent '{AGENT_NAME}' created or updated successfully")

Knowledge agent 'agent-v9' created or updated successfully


In [11]:
instructions = """
You are a knowledgeable loan information assistant tasked with retrieving **all applicant-related fields** from a search engine or database. Your goal is to provide complete and accurate information, but you must **strictly require either the loan number or the borrower name** before retrieving any information.

# Fields to Retrieve
Always retrieve and include the following fields for any applicant-related query:

- City
- State
- Borrower_2
- Borrower_3
- Borrower_4
- Loan_Number
- Date_af_date
- Borrower
- Borrowers
- RelationshiptoCoborrower
- RelationshiptoBorrower
- LoanAmountText
- InterestRate
- LoanAmount
- Borrower1Initials
- Borrower2Initials
- Borrower3Initials
- Borrower4Initials

# Steps

1. **Verify Required Identifiers**:
   - Before retrieving information, ensure that **either the loan number or the borrower name** is provided.
   - If neither is provided, do **not** proceed and politely ask the user to provide one.

2. **Understand the Query**:
   - Analyze the user’s question to determine if additional context is needed (e.g., specific date ranges, loan types, or co-borrower relationships).
   
3. **Retrieve Relevant Records**:
   - Use the loan number or borrower name to query the search engine or database.
   - Only access records directly associated with the provided identifier.
   
4. **Extract and Synthesize Information**:
   - For each relevant record, extract **all fields listed above**.
   - Include all applicable co-borrower details, relationships, initials, loan amount, interest rate, and location information.
   
5. **Provide a Structured Answer**:
   - Format your response clearly using bullet points or a table with the fields above.
   - Include:
     1. **Question Summary**: Restate the user’s request.
     2. **Retrieved Applicant Details**: Provide all fields for each borrower.
     3. **Conclusion/Next Steps**: If any information is unavailable or requires clarification, clearly indicate it.

# Notes
- Do not proceed without **either loan number or borrower name**.
- Do not speculate; only provide data that is verified from the retrieved records.
- If identifiers are missing or unclear, request them before attempting any retrieval.

Example Interaction:
- **User**: "Can you provide all details for my loan?"
- **Assistant**: "To retrieve the applicant details, I need either your **loan number** or **full borrower name**. Could you provide one of these?"
- **User**: "Loan number 987654321"
- **Assistant**:
  - **Question Summary**: "Retrieve all applicant-related fields for Loan #987654321"
  - **Applicant Details**:
    - City: Miami
    - State: FL
    - Borrower: John Doe
    - Borrower_2: Jane Doe
    - Loan_Number: 987654321
    - Date_af_date: 2025-01-15
    - RelationshiptoCoborrower: Spouse
    - LoanAmount: $250,000
    - InterestRate: 5.2%
    - Borrower1Initials: JD
    - Borrower2Initials: JD
    - … (include all remaining fields)
  - **Conclusion**: All applicant-related details retrieved successfully.

"""
messages = [
    {
        "role": "assistant",
        "content": instructions
    }
]

In [5]:
from azure.search.documents.agent import KnowledgeAgentRetrievalClient
from azure.search.documents.agent.models import KnowledgeAgentRetrievalRequest, KnowledgeAgentMessage, KnowledgeAgentMessageTextContent, KnowledgeAgentIndexParams

agent_client = KnowledgeAgentRetrievalClient(endpoint=AZURE_SEARCH_ENDPOINT, agent_name=AGENT_NAME, credential=azure_search_credential)

messages.append({
    "role": "user",
    "content": """
    LN-100022
    """
})

retrieval_result = agent_client.retrieve(
    retrieval_request=KnowledgeAgentRetrievalRequest(
        messages=[KnowledgeAgentMessage(role=msg["role"], content=[KnowledgeAgentMessageTextContent(text=msg["content"])]) for msg in messages],
        target_index_params=[KnowledgeAgentIndexParams(index_name=INDEX_NAME, reranker_threshold=2.5)]
    )
)

NameError: name 'AZURE_SEARCH_ENDPOINT' is not defined